<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB18_Case_Study_ECMWF_Predicting_Wave_Height_from_Wind_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB18 · Clase 18 — Caso de estudio: datos meteorológicos ECMWF, prediciendo la altura de ola a partir del viento**

## Bloque 4: Proyectos — Casos de estudio (apertura)

Los bloques 2–3 enseñaron las herramientas; el Bloque 4 las aplica a casos de estudio reales completos — más cerca de cómo usarás realmente este material después del curso. Este primer caso de estudio usa **datos reales de reanálisis ECMWF**, obtenidos en vivo del mismo Copernicus Climate Data Store que usan los profesionales en su trabajo operativo.

**El problema del mundo real**: las olas del mar se generan por el viento — lo grandes que llegan a ser depende de la velocidad del viento, cuánto tiempo lleva soplando (duración), y cuánta agua abierta ha recorrido (fetch). Si un modelo basado en datos puede aprender esa relación física real a partir de datos reales, se vuelve genuinamente útil: `una forma rápida y barata de estimar el estado del mar solo a partir del viento`, valiosa siempre que ejecutar un modelo de oleaje completo basado en física (como WAM o WaveWatch III) no sea práctico — una estimación rápida a bordo con conectividad limitada, cubrir un hueco donde no hay salida disponible de un modelo de oleaje todavía, o comprobar la coherencia de la salida de un modelo más costoso.

Esta clase tiene **dos partes**, y la diferencia entre ellas importa:
- **Parte A** (Secciones 3–10) valida un enfoque de modelado usando datos reales de ERA5 para **una fecha del calendario — el 15 de enero — agrupada en cinco años reales (2019–2023)**, donde ya conocemos la altura de ola real en todas partes. Usar varios años de la *misma* fecha, en vez de una única instantánea, importa: la altura de ola depende mucho de la época del año, así que agrupar varios años de la misma fecha enseña al modelo el rango real de condiciones de viento-oleaje invernales en esta ubicación, en vez de memorizar el tiempo de un único día. Este es el mismo planteamiento de "etiqueta conocida" que el modelo de consumo de combustible de `NB07` o el modelo de resistencia de casco de `NB10`, usado para construir y comprobar la coherencia de un método *antes* de confiar en él sobre algo genuinamente desconocido.
- **Parte B** (Sección 11) es la prueba real: aplicar ese enfoque ya validado al **15 de enero de 2024 — la misma fecha del calendario, pero un año que el modelo nunca ha visto** — prediciendo la altura de ola a partir del viento **únicamente**, y revelando los valores reales solo después para comprobarlo. Reservar un *año*, no una época del año distinta, es lo que hace que esto sea una prueba justa: comprueba si el modelo generaliza a un tiempo genuinamente nuevo (un año que nunca vio), sin pedirle además que generalice a través de una época del año para la que nunca fue entrenado.

Elegirás el enfoque de modelado tú mismo en la Parte A, usando el marco de decisión de `NB17` — esta clase no te dice qué arquitectura usar.

> **Importante — hazlo antes de clase**: regístrate en una cuenta gratuita de Copernicus CDS y genera tu clave de API en [cds.climate.copernicus.eu/profile](https://cds.climate.copernicus.eu/profile) (se recomienda un correo institucional). El registro/aprobación puede tardar un poco, así que hazlo **antes** de la sesión, no durante ella.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar, físicamente, por qué el viento puede predecir la altura de ola, y por qué esa relación tiene valor operativo real.
- Explicar por qué una prueba justa de generalización debe reservar un *año*, no una época del año distinta, cuando la variable objetivo es estacional.
- Obtener datos reales de reanálisis multi-año del Copernicus Climate Data Store mediante `cdsapi`.
- Reorganizar datos climáticos NetCDF en rejilla, agrupados a través de varios años, en una tabla plana lista para ML.
- Aplicar el marco de decisión de `NB17` a un problema nuevo y justificar una elección de modelado.
- Distinguir un ejercicio de validación de metodología (etiquetas conocidas, años de entrenamiento) de una predicción genuina sobre un año reservado, y explicar por qué un proyecto real necesita ambos.
- Aplicar un modelo entrenado para predecir la misma fecha del calendario en un año que nunca ha visto — sin mirar sus valores reales hasta después de predecir — y evaluarlo con honestidad.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso, introducción al Bloque 4, hoja de ruta de hoy | 5 min | Teoría |
| 2 | Qué es ECMWF/ERA5, y por qué "viento → altura de ola" es un problema real | 10 min | Teoría |
| 3 | Configurar el acceso a la API de Copernicus CDS | 10 min | Práctica |
| 4 | Descargar datos reales de ERA5 multi-año | 5 min | Práctica |
| 5 | Explorar la rejilla NetCDF | 5 min | Práctica |
| 6 | Reorganizar y agrupar cinco años en una tabla plana | 10 min | Práctica |
| 7 | Explorar el dataset real ya aplanado | 15 min | Práctica |
| 8 | Aplicar el marco de decisión de `NB17` | 5 min | Teoría + Práctica |
| 9 | Práctica: entrenar y comparar modelos (validación de metodología) | 15 min | Práctica |
| 10 | Evaluación y una comprobación espacial de coherencia (validación de metodología) | 15 min | Práctica |
| 11 | Una prueba genuinamente predictiva: predecir un año reservado | 20 min | Práctica |
| 12 | Resumen, tarea, qué sigue en el Bloque 4 | 5 min | Teoría |

> Los tiempos son una orientación aproximada, no un guion estricto — no hay descansos programados. Si cubrimos todo con tiempo de sobra, la clase termina antes; eso puede pasar y está bien.

---

## 1. Repaso e introducción al Bloque 4

- **Bloque 2** (`NB02`–`NB10`): el conjunto de herramientas clásico de ML.
- **Bloque 3** (`NB11`–`NB17`): Deep Learning, cerrando con un marco de decisión para elegir entre todo lo aprendido hasta ahora.
- **Bloque 4** (empieza hoy): casos de estudio completos, aplicando ese conjunto de herramientas a problemas reales nuevos.

El Bloque 4 es también donde la evaluación del curso alcanza a la docencia: tu **proyecto final individual** (40% de la nota — un caso de estudio no visto que presentas y defiendes) usa un dataset real *distinto* de los enseñados en clase, y la **entrega de caso de estudio** (20%) te permite elegir y volver a entregar tu propia resolución de *cualquier* notebook trabajado en clase, incluido este. A partir de aquí, algunas sesiones de clase serán casos de estudio nuevos como este, y otras serán tiempo supervisado para que trabajes en tu propio proyecto, con un breve punto de control que entregar cada sesión — pregunta a tu profesor por el calendario exacto.

---

## 2. Qué es ECMWF/ERA5, y por qué "viento → altura de ola" es un problema real

El **[Centro Europeo de Previsiones Meteorológicas a Plazo Medio (ECMWF)](https://en.wikipedia.org/wiki/ECMWF)** es una organización intergubernamental y uno de los centros líderes del mundo en predicción numérica del tiempo. Su dataset **[ERA5](https://en.wikipedia.org/wiki/ERA5)** es un *reanálisis*: ni una observación en bruto ni una predicción, sino una reconstrucción físicamente coherente de las condiciones atmosféricas y de superficie oceánica pasadas, producida combinando millones de observaciones históricas reales con un modelo numérico del tiempo. En la práctica, esto significa que ERA5 nos da **datos realistas y físicamente coherentes de tiempo y estado del mar para cualquier lugar y momento desde 1940** — exactamente el tipo de datos ambientales reales que un ingeniero naval u oceánico usaría para planificación de rutas, estimaciones de carga estructural, o análisis histórico del tiempo.

### Por qué "viento → altura de ola" en concreto

`Las olas se forman porque el viento transfiere energía a la superficie del mar`. Lo grandes que llegan a ser depende de tres factores físicos reales: la **velocidad** del viento, cuánto tiempo lleva soplando (**duración**), y cuánta agua abierta ha recorrido (**fetch**). Esto es oceanografía física establecida — no una coincidencia elegida para dar un problema de clase bien empaquetado.

Un modelo que aprenda bien esta relación a partir de datos reales tiene un valor operativo genuino. Los modelos de oleaje completos basados en física (como WAM o WaveWatch III) son el estándar preciso y de confianza, pero son computacionalmente costosos y necesitan infraestructura dedicada para ejecutarse — no son algo que se vuelva a ejecutar casualmente para una estimación rápida. Un modelo ligero, basado en datos, de viento→oleaje nunca puede sustituirlos para una predicción seria, pero es útil en cualquier sitio donde una estimación rápida y aproximada sea mejor que ninguna estimación: una comprobación rápida a bordo con conectividad limitada, cubrir un hueco donde la salida de un modelo de oleaje no está disponible para un momento o lugar dado, o comprobar la coherencia de la salida de un modelo más costoso antes de confiar en ella.

### Un detalle metodológico que merece explicarse antes de cualquier código

La altura de ola depende fuertemente de la época del año — una tormenta de enero y un día en calma de julio no son el mismo régimen físico, y un modelo entrenado en una época no tiene una base real para predecir otra distinta; comprobar eso solo nos diría lo obvio (las épocas del año difieren), no si el modelo realmente generaliza. Para comprobar la generalización genuina de año a año *sin* cruzar también épocas del año, esta clase fija la fecha del calendario — el 15 de enero — y varía solo el **año**: entrenando con varios años pasados de esa fecha, y prediciendo después la misma fecha en un año nunca visto. Eso aísla la pregunta que realmente nos interesa: ¿generaliza el modelo a un *tiempo* nuevo (un año que no ha visto), manteniendo constante la *época del año*?

Esa es la pregunta real que responde esta clase — no "¿podemos ajustar una curva a unos números?", sino "¿aparece una relación real y físicamente motivada con la claridad suficiente en datos reales como para que un modelo la aprenda, y puede ese modelo decir después algo verdadero sobre un año que nunca ha visto?" La Parte A de abajo construye y valida el enfoque, agrupando varios inviernos reales; la Parte B (Sección 11) es donde realmente lo respondemos, sobre un año genuinamente reservado.

---

## 3. Configurar el acceso a la API de Copernicus CDS

Con tu clave de API gratuita de [cds.climate.copernicus.eu/profile](https://cds.climate.copernicus.eu/profile) ya lista, instala el cliente y guarda tus credenciales para esta sesión:

In [ ]:
%pip install -q cdsapi xarray netCDF4 cartopy

Ejecuta la celda de abajo y pega tu clave **cuando se te pida** — usa `getpass`, que oculta lo que escribes/pegas y, más importante aún, significa que tu clave real **nunca se escribe en el código de este notebook ni en su salida guardada**. A diferencia de una clave escrita directamente en una celda (que es exactamente cómo se filtró una clave de API real en este repositorio antes en la historia de este curso — mira las primerísimas correcciones de seguridad de este proyecto), un prompt de `getpass` solo existe en la memoria de la sesión en ejecución; nada de eso acaba en este fichero cuando lo guardas, lo compartes, o lo subes a GitHub.

1. Ve a [cds.climate.copernicus.eu/profile](https://cds.climate.copernicus.eu/profile) e inicia sesión.
2. Copia tu **Personal Access Token** (una cadena larga de letras, números y guiones).
3. Ejecuta la celda de abajo. Aparece un cuadro de entrada oculto — pega ahí tu token y pulsa Intro.

In [ ]:
import os
from getpass import getpass

CDS_API_KEY = getpass("Paste your CDS API key from https://cds.climate.copernicus.eu/profile (input hidden): ").strip()

if not CDS_API_KEY:
    raise ValueError(
        "No key entered. Get one from https://cds.climate.copernicus.eu/profile "
        "and re-run this cell."
    )

cdsapirc = f"url: https://cds.climate.copernicus.eu/api\nkey: {CDS_API_KEY}\n"

with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write(cdsapirc)

print("CDS API key saved for this session (not stored anywhere in this notebook).")

---

## 4. Descargar datos reales de ERA5 multi-año

Solicita las componentes del viento a 10 m y la altura significativa de ola, sobre el Atlántico Norte/la plataforma de Europa Occidental — una región real y relevante para lo naval (el Canal de la Mancha, el Golfo de Vizcaya, el Mar del Norte) — para el **15 de enero, 12:00 UTC, a través de cinco años reales (2019–2023)**. La API del CDS acepta una lista de años en una única solicitud, así que esto sigue siendo una sola descarga, solo que abarca varios inviernos reales en vez de un único día:

In [ ]:
import cdsapi

client = cdsapi.Client()

TRAINING_YEARS = ["2019", "2020", "2021", "2022", "2023"]

client.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "format": "netcdf",
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "significant_height_of_combined_wind_waves_and_swell",
        ],
        "year": TRAINING_YEARS,
        "month": "01",
        "day": "15",
        "time": ["12:00"],
        "area": [60, -20, 35, 10],  # North, West, South, East
    },
    "era5_training_years.nc",
)

El Climate Data Store a veces devuelve un único fichero NetCDF y a veces un archivo zip que separa las variables por "stream" interno de datos — maneja ambos casos para que el resto del notebook no dependa de cuál te haya tocado:

In [ ]:
import zipfile
import glob

target = "era5_training_years.nc"
extract_dir = "."

if zipfile.is_zipfile(target):
    extract_dir = "era5_extracted"
    with zipfile.ZipFile(target) as zf:
        zf.extractall(extract_dir)
    print("Zip archive detected and extracted:", os.listdir(extract_dir))
else:
    print("Single NetCDF file, no extraction needed.")

nc_files = glob.glob(os.path.join(extract_dir, "*.nc")) or [target]
print("NetCDF files:", nc_files)

---

## 5. Explorando la rejilla NetCDF

Abre todos los ficheros encontrados e identifica cuál contiene la variable de oleaje (`swh`) y cuál las componentes del viento (`u10`/`v10`) — esto también protege al notebook frente a que el/los fichero(s) vengan en una disposición distinta de la esperada:

In [ ]:
import xarray as xr

datasets = [xr.open_dataset(f) for f in nc_files]
for i, ds in enumerate(datasets):
    print(f"File {i}: variables = {list(ds.data_vars)}, dims = {dict(ds.sizes)}")

wave_ds = next(ds for ds in datasets if "swh" in ds.data_vars)
wind_ds = next(ds for ds in datasets if "u10" in ds.data_vars and "v10" in ds.data_vars)

Dibuja la rejilla de altura de ola en bruto para un año de entrenamiento de ejemplo — la misma primera comprobación de coherencia que merece cualquier dataset real en rejilla antes de modelar nada. Los otros cuatro años se agrupan en silencio por ahora y tienen su turno en la Parte 6:

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

example_year_idx = 0  # first of the 5 training years
swh_example = wave_ds["swh"].isel(valid_time=example_year_idx)
print("Showing:", str(wave_ds.valid_time.isel(valid_time=example_year_idx).values)[:10])

lon_min, lon_max = float(swh_example.longitude.min()), float(swh_example.longitude.max())
lat_min, lat_max = float(swh_example.latitude.min()), float(swh_example.latitude.max())

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

im = ax.imshow(swh_example.values, origin="upper", extent=[lon_min, lon_max, lat_min, lat_max],
                cmap="viridis", transform=ccrs.PlateCarree())

ax.coastlines(resolution="110m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.gridlines(draw_labels=True)

plt.colorbar(im, ax=ax, orientation="vertical", pad=0.05, shrink=0.7, label="Significant wave height (m)")
plt.title("Real ERA5 significant wave height -- one example training year")
plt.tight_layout()
plt.show()

Con las líneas de costa y fronteras reales dibujadas, la región en blanco/NaN se alinea visiblemente con tierra (Francia, el Reino Unido, España, Portugal) — la altura de ola solo está definida sobre agua abierta, lo cual será relevante en un momento. Poder ver *dónde* en la Tierra están realmente estos datos `importa para un caso de estudio naval/oceánico, no solo como decoración`: es cómo reconocerías más adelante un mar cerrado limitado por fetch (el Canal de la Mancha, el Mar del Norte) frente al oleaje abierto del Atlántico en la Parte 7.

---

## 6. Reorganizar y agrupar cinco años en una tabla plana

Todos los modelos que hemos usado desde `NB02` esperan una tabla plana: una fila por ejemplo, una columna por característica. Una rejilla espacial necesita reorganizarse primero — cada celda `(latitude, longitude)` se convierte en una fila, y como ahora tenemos **5 años de entrenamiento**, la rejilla de cada año se reorganiza y se apila en una única tabla combinada. Deliberadamente, no se mantiene ninguna columna `year` como característica: queremos que el modelo aprenda la relación general viento→oleaje para esta época del año, no que memorice un desplazamiento propio de cada año.

Un detalle real: los parámetros de oleaje de ERA5 (como `swh`) y sus parámetros de superficie/atmosféricos (como `u10`/`v10`) no siempre se entregan en rejillas de forma idéntica, incluso cuando se solicitan juntos sobre la misma área — una peculiaridad conocida de cómo el sistema de predicción de ECMWF representa internamente el oleaje oceánico. Comprueba primero el tamaño de rejilla de ambos datasets:

In [ ]:
print("wave_ds grid:", wave_ds.sizes)
print("wind_ds grid:", wind_ds.sizes)

Si los dos tamaños de arriba difieren, el campo de viento necesita interpolarse sobre la rejilla del campo de oleaje antes de que puedan compartir una tabla — hecho automáticamente abajo tanto si coincidían como si no, así que este notebook funciona en cualquier caso:

In [ ]:
import numpy as np
import pandas as pd

# Align the wind field onto the wave field's exact grid, whether or not they
# originally matched -- interpolation is a no-op if the grids already agree.
wind_ds_aligned = wind_ds.interp(latitude=wave_ds.latitude, longitude=wave_ds.longitude)
lat_grid, lon_grid = np.meshgrid(wave_ds.latitude.values, wave_ds.longitude.values, indexing="ij")

n_years = wave_ds.sizes["valid_time"]
print(f"Pooling {n_years} training years into one table...")

year_frames = []
for t in range(n_years):
    u10 = wind_ds_aligned["u10"].isel(valid_time=t).values
    v10 = wind_ds_aligned["v10"].isel(valid_time=t).values
    swh_vals = wave_ds["swh"].isel(valid_time=t).values

    assert u10.shape == swh_vals.shape == lat_grid.shape, (
        f"Grid shapes still don't match for year index {t}: u10 {u10.shape}, "
        f"swh {swh_vals.shape}, lat_grid {lat_grid.shape} -- inspect wave_ds/wind_ds.sizes above."
    )

    wind_speed = np.sqrt(u10 ** 2 + v10 ** 2)
    wind_direction = (np.degrees(np.arctan2(u10, v10)) + 360) % 360

    year_frames.append(pd.DataFrame({
        "latitude": lat_grid.ravel(),
        "longitude": lon_grid.ravel(),
        "u10": u10.ravel(),
        "v10": v10.ravel(),
        "wind_speed": wind_speed.ravel(),
        "wind_direction": wind_direction.ravel(),
        "swh": swh_vals.ravel(),
    }))

grid_df = pd.concat(year_frames, ignore_index=True).dropna()
print(grid_df.shape)
grid_df.head()

`dropna()` eliminó de un solo paso cada celda de rejilla en tierra a través de los 5 años — `una razón real y físicamente significativa para los datos ausentes`, distinta en su naturaleza del desajuste de ficheros de sensor de `NB13` pero manejada del mismo modo: entender *por qué* falta antes de decidir qué hacer al respecto. `grid_df` ahora contiene aproximadamente 5 veces las celdas oceánicas que daría una única instantánea — la relación viento/oleaje del mismo lugar, muestreada a través de 5 inviernos reales en vez de uno.

---

## 7. Explorando el dataset real ya aplanado

Las tres preguntas iniciales de `NB02`, una vez más, sobre este dataset real, multi-año y agrupado:

In [ ]:
grid_df.describe()

Y la relación de la que depende toda la clase — ¿la altura de ola realmente sigue a la velocidad del viento en estos datos reales?

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(grid_df["wind_speed"], grid_df["swh"], alpha=0.3, s=10)
plt.xlabel("Wind speed (m/s)")
plt.ylabel("Significant wave height (m)")
plt.title("Wind speed vs. wave height, every real grid cell")
plt.show()

**Pruébalo tú mismo**: pon un número a lo que muestra el diagrama de dispersión — calcula la correlación real entre `wind_speed` y `swh` a través de cada celda de rejilla real agrupada.

In [ ]:
correlation = grid_df[["wind_speed", "swh"]].corr().iloc[0, 1]
print(f"Correlation between wind speed and wave height: {correlation:.3f}")


**Lee tu propio gráfico**: ¿es la relación una línea limpia, una tendencia ruidosa, o casi ninguna relación en absoluto? Los mares reales limitados por fetch (un mar cerrado como partes del Canal de la Mancha, donde el viento no tiene espacio para generar olas grandes) pueden verse bastante distintos del oleaje abierto del Atlántico en la misma instantánea — `latitude`/`longitude` puede acabar importando tanto como la propia velocidad del viento.

---

## 8. Aplicando el marco de decisión de `NB17`

Antes de escribir ningún código de modelo, pasa este problema real por las tres preguntas de `NB17`:

1. **¿Etiquetas?** Sí — `swh` es un objetivo real y conocido para cada celda de rejilla, en cada año de entrenamiento.
2. **¿Forma de los datos?** Tabular — cada fila son las características de una única celda de rejilla, no una imagen ni una secuencia (aunque se originó en una rejilla espacial, ya la hemos aplanado).
3. **¿Volumen de datos?** Varios miles de celdas de rejilla oceánicas tras `dropna()`, ahora agrupadas a través de 5 años — moderado, no enorme, pero mayor de lo que daría cualquier instantánea única.

El marco de `NB17`, aplicado con honestidad, apunta hacia el **ML clásico** como al menos una base sólida aquí — es también, de forma plausible, la respuesta final correcta, la misma expectativa que el propio experimento cara a cara de `NB17` planteó sobre los datos del yate de `NB10`, donde `NB17` deja deliberadamente que seas tú quien confirme al ganador real ejecutando los números en vez de darlo por sentado. Una red neuronal sigue siendo una elección legítima para *también* probar y comparar, pero el marco no da ninguna razón para asumir que va a ganar automáticamente. **Tu tarea**: elige al menos un modelo clásico y justifica tu elección en voz alta (a un compañero o a tu profesor) antes de escribir el código de entrenamiento de abajo.

---

## 9. Práctica: entrenar y comparar modelos

Primero, una comprobación de fuga de datos — `wind_speed` y `wind_direction` se *derivaron* ambas de `u10`/`v10`, así que usar las cuatro juntas simplemente estaría dándole al modelo la misma información dos veces en formas distintas, el mismo principio de redundancia del ejemplo de `CO2_emissions` de `NB07`. Usa `u10`/`v10` **o** `wind_speed`/`wind_direction`, no ambas representaciones a la vez:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = ["latitude", "longitude", "wind_speed", "wind_direction"]
X = grid_df[feature_cols]
y = grid_df["swh"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

> **Una limitación real que merece nombrarse, no esconderse**: esto es una partición *aleatoria* tanto de datos espacialmente correlacionados (celdas de rejilla cercanas tienen alturas de ola parecidas) como, ahora, de ubicaciones repetidas a través de los años (la misma celda `(latitude, longitude)` aparece una vez por año de entrenamiento, y sus 5 versiones tampoco son del todo independientes entre sí). `Una evaluación más estricta reservaría toda una región espacial, o todo un año, en vez de filas aleatorias` (pruébalo de las dos formas como tarea). Seguimos con la partición aleatoria para la comprobación de metodología de hoy, etiquetada con honestidad como una simplificación — la prueba del año reservado de la Parte 11 es exactamente la versión más estricta de esto, hecha correctamente.

Compara una base lineal frente a un ensemble de árboles — los candidatos clásicos que favorecía el marco de `NB17`:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

candidate_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in candidate_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring="r2")
    cv_results[name] = scores

pd.DataFrame(cv_results).mean().sort_values(ascending=False)

**Pruébalo tú mismo**: añade un Gradient Boosting Regressor (el tercer candidato de `NB10`) a la comparación de arriba — ¿supera aquí tanto a la regresión lineal como al Random Forest?

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

candidate_models["Gradient Boosting"] = GradientBoostingRegressor(random_state=42)
gb_scores = cross_val_score(candidate_models["Gradient Boosting"], X_train_scaled, y_train, cv=cv, scoring="r2")
cv_results["Gradient Boosting"] = gb_scores

pd.DataFrame(cv_results).mean().sort_values(ascending=False)


---

## 10. Evaluación y una comprobación espacial de coherencia

Esto todavía evalúa sobre filas extraídas del *mismo* conjunto de años de entrenamiento (2019–2023) — una comprobación justa del enfoque de modelado, pero todavía no una prueba genuina de predecir un año nuevo (eso es la Sección 11, justo después de esta). Ajusta el candidato más fuerte sobre el conjunto de entrenamiento completo y evalúa una sola vez sobre el conjunto de test intacto:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_model = RandomForestRegressor(n_estimators=200, random_state=42)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)

print(f"MAE:  {mean_absolute_error(y_test, y_pred):.3f} m")
print(f"RMSE: {mean_squared_error(y_test, y_pred) ** 0.5:.3f} m")
print(f"R2:   {r2_score(y_test, y_pred):.3f}")

**Pruébalo tú mismo**: ¿de qué característica depende más el Random Forest — latitud, longitud, velocidad del viento, o dirección del viento?

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances


Una comprobación espacial de coherencia — dibuja los residuos del conjunto de test de vuelta sobre el mapa. Errores dispersos al azar sugieren un modelo razonablemente sin sesgo; errores agrupados en una región (digamos, el Canal de la Mancha en concreto) sugerirían que al modelo se le escapa sistemáticamente algo sobre el estado del mar de esa zona:

In [ ]:
residuals = y_test.values - y_pred

fig = plt.figure(figsize=(9, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

sc = ax.scatter(X_test["longitude"], X_test["latitude"], c=residuals, cmap="coolwarm",
                 vmin=-abs(residuals).max(), vmax=abs(residuals).max(), s=15,
                 transform=ccrs.PlateCarree())

ax.coastlines(resolution="110m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.gridlines(draw_labels=True)

plt.colorbar(sc, ax=ax, orientation="vertical", pad=0.05, shrink=0.7, label="Residual (actual - predicted), m")
plt.title("Where does the model's error concentrate?")
plt.tight_layout()
plt.show()

---

## 11. Una prueba genuinamente predictiva: predecir un año reservado

Las Secciones 5–10 agruparon cinco inviernos reales (2019–2023) de la *misma* fecha del calendario — el 15 de enero — para validar un enfoque de modelado. Esa es una comprobación de metodología justa, y agrupar años (en vez de una única instantánea) ya le da al modelo exposición a la variabilidad real de año a año en las condiciones invernales de viento-oleaje. Pero cada uno de esos años estaba disponible durante el entrenamiento; todavía no hemos probado el modelo sobre un año que genuinamente nunca vio.

Ahora hacemos lo real: obtenemos el **15 de enero de 2024** — la misma fecha del calendario, deliberadamente, para probar la generalización a través de *años*, no a través de *estaciones* — predecimos sus alturas de ola a partir del viento **únicamente**, y solo revelamos los valores reales después para comprobarlo. Esto refleja cómo se usaría realmente un modelo en operación: entrenado una vez con inviernos históricos, y aplicado después a un invierno nuevo cuyo resultado todavía no se conoce.

In [ ]:
client.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "format": "netcdf",
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "significant_height_of_combined_wind_waves_and_swell",
        ],
        "year": "2024",
        "month": "01",
        "day": "15",
        "time": ["12:00"],
        "area": [60, -20, 35, 10],
    },
    "era5_forecast_target.nc",
)

Este fichero técnicamente también contiene las alturas de ola reales del `2024-01-15` — pero para que esto siga siendo una prueba genuina, **no abriremos la variable `swh` hasta después de hacer nuestra predicción**. Desde aquí hasta el paso de revelación, trata esto como si solo hubiera datos de viento disponibles:

In [ ]:
target2 = "era5_forecast_target.nc"
extract_dir2 = "."

if zipfile.is_zipfile(target2):
    extract_dir2 = "era5_forecast_extracted"
    with zipfile.ZipFile(target2) as zf:
        zf.extractall(extract_dir2)

nc_files2 = glob.glob(os.path.join(extract_dir2, "*.nc")) or [target2]

datasets2 = [xr.open_dataset(f) for f in nc_files2]
wind_ds2 = next(ds for ds in datasets2 if "u10" in ds.data_vars and "v10" in ds.data_vars)
wave_ds2 = next(ds for ds in datasets2 if "swh" in ds.data_vars)  # not opened/used yet

print("New date's wind grid:", wind_ds2.sizes)

Construye la tabla de características exactamente como en la Parte 6 — pero necesitamos una forma de saber qué celdas de rejilla son océano (frente a tierra, donde no existe altura de ola) *sin* mirar el `swh` de este año nuevo. La tierra y el mar no se mueven de un año a otro, así que reutilizamos las ubicaciones de celdas oceánicas ya establecidas a partir de los años de entrenamiento agrupados en la Parte 6, puramente por geografía:

In [ ]:
wind_ds2_aligned = wind_ds2.interp(latitude=wave_ds2.latitude, longitude=wave_ds2.longitude)
u10_2 = wind_ds2_aligned["u10"].isel(valid_time=0).values
v10_2 = wind_ds2_aligned["v10"].isel(valid_time=0).values
lat_grid2, lon_grid2 = np.meshgrid(wave_ds2.latitude.values, wave_ds2.longitude.values, indexing="ij")

wind_speed2 = np.sqrt(u10_2 ** 2 + v10_2 ** 2)
wind_direction2 = (np.degrees(np.arctan2(u10_2, v10_2)) + 360) % 360

target_df = pd.DataFrame({
    "latitude": lat_grid2.ravel(),
    "longitude": lon_grid2.ravel(),
    "wind_speed": wind_speed2.ravel(),
    "wind_direction": wind_direction2.ravel(),
})

# Ocean/land is geography, not weather -- reuse the cells already known to be
# ocean from Part 6's training snapshot, without looking at this date's swh.
ocean_cells = grid_df[["latitude", "longitude"]].drop_duplicates()
target_df = target_df.merge(ocean_cells, on=["latitude", "longitude"], how="inner")
print(target_df.shape)

Predice, usando el modelo ya entrenado en la Parte 9 — sin reentrenar, sin hacer trampas:

In [ ]:
X_target = target_df[feature_cols]
X_target_scaled = scaler.transform(X_target)
target_df["predicted_swh"] = best_model.predict(X_target_scaled)
target_df.head()

**Ahora, y solo ahora**, revela las alturas de ola reales para esta fecha, para ver qué tal ha ido realmente la predicción:

In [ ]:
swh2 = wave_ds2["swh"].isel(valid_time=0)
lat_grid2b, lon_grid2b = np.meshgrid(wave_ds2.latitude.values, wave_ds2.longitude.values, indexing="ij")
truth_df = pd.DataFrame({
    "latitude": lat_grid2b.ravel(),
    "longitude": lon_grid2b.ravel(),
    "true_swh": swh2.values.ravel(),
})

comparison = target_df.merge(truth_df, on=["latitude", "longitude"], how="left")

print(f"MAE:  {mean_absolute_error(comparison['true_swh'], comparison['predicted_swh']):.3f} m")
print(f"RMSE: {mean_squared_error(comparison['true_swh'], comparison['predicted_swh']) ** 0.5:.3f} m")
print(f"R2:   {r2_score(comparison['true_swh'], comparison['predicted_swh']):.3f}")

**Pruébalo tú mismo**: MAE y RMSE están en metros — pon el error del año reservado también en términos relativos, como un porcentaje de la altura de ola media real, para una lectura de cuán grande es realmente el fallo, independiente de la escala.

In [ ]:
mean_true_swh = comparison["true_swh"].mean()
mae_pct = mean_absolute_error(comparison["true_swh"], comparison["predicted_swh"]) / mean_true_swh * 100

print(f"Mean true wave height: {mean_true_swh:.2f} m")
print(f"MAE as % of mean wave height: {mae_pct:.1f}%")


**Compara estos números con los de la Parte 10.** ¿Son parecidos, o notablemente peores? Como la Parte 11 reserva un *año*, no una *estación*, esta es una comparación justa y equivalente: cualquier bajada aquí refleja variabilidad real de año a año en las condiciones invernales (un invierno más tormentoso o más tranquilo de lo habitual entre 2019–2023), no que se le esté pidiendo al modelo generalizar a través de un régimen climático completamente distinto que nunca vio. Eso hace que este resultado sea directamente fiable para la pregunta que un ingeniero naval realmente haría: "si entrené esto con inviernos pasados, ¿cuánto me fiaría de ello en uno nuevo?" Una comparación visual lado a lado lo hace concreto:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), subplot_kw={"projection": ccrs.PlateCarree()})

for ax, col, title in zip(axes, ["true_swh", "predicted_swh"], ["True (revealed)", "Predicted"]):
    sc = ax.scatter(comparison["longitude"], comparison["latitude"], c=comparison[col],
                     cmap="viridis", vmin=comparison["true_swh"].min(), vmax=comparison["true_swh"].max(),
                     s=15, transform=ccrs.PlateCarree())
    ax.coastlines(resolution="110m")
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    ax.set_title(f"{title} wave height, 2024-01-15")

fig.colorbar(sc, ax=axes, orientation="horizontal", pad=0.08, shrink=0.6, label="Significant wave height (m)")
plt.show()

Esta es la prueba real que este notebook se propuso ejecutar: `no ajustar un modelo a datos que ya puede ver`, sino predecir un **año** genuinamente nuevo — con la estación mantenida constante — y comprobarlo solo después. Esa es la respuesta honesta y completa a "¿puede el viento por sí solo predecir la altura de ola?", y es una prueba justa precisamente porque tuvimos cuidado con *qué* reservamos.

---

## 12. Resumen, tarea, y qué sigue en el Bloque 4

- El reanálisis ERA5 proporciona datos históricos reales y físicamente coherentes de tiempo/océano, obtenidos en vivo mediante la API de Copernicus CDS — un flujo de trabajo profesional genuino, no un atajo docente.
- El viento predice la altura de ola porque genera físicamente las olas (velocidad, duración, fetch) — una relación real con valor operativo real como sustituto barato de los modelos de oleaje costosos basados en física.
- La altura de ola es estacional, así que una prueba justa de generalización debe reservar un *año*, no una época del año distinta — agrupar varios años de la misma fecha del calendario para entrenar, y probar después sobre un año reservado, aísla la variabilidad real de año a año del efecto estacional, mucho mayor (e injusto de probar).
- Los datos NetCDF en rejilla se reorganizan en una tabla plana exactamente igual que cualquier otro dataset en cuanto entiendes sus dimensiones — `dropna()` eliminando las celdas de tierra fue un paso de limpieza físicamente significativo, no arbitrario.
- El marco de decisión de `NB17`, aplicado a un problema genuinamente nuevo, apuntó hacia el ML clásico — y una comparación rápida con validación cruzada lo confirmó.
- Evaluar sobre filas reservadas del *mismo* conjunto de años de entrenamiento (Parte 10) valida un método; predecir un *año* genuinamente nuevo (Parte 11) — nunca visto, con tierra/océano determinados solo por geografía, valores reales revelados solo después de predecir — es la prueba real y honesta de generalización.
- Un mapa de residuos y un mapa lado a lado de predicho frente a real son herramientas de interpretación específicas de datos espaciales, junto con los habituales MAE/RMSE/R².

### Qué sigue en el Bloque 4

Las próximas sesiones impartidas de este bloque cubren otros casos de estudio reales (registros históricos de travesías, datos de terreno/elevación). Las sesiones *entre* ellas son tu tiempo supervisado de proyecto — trae progreso real y verificable en tu proyecto final individual cada vez, según el calendario de la rúbrica de `Final_Project_Wave_Height_Forecasting_STARTER.ipynb` (nota: ese proyecto usa un dataset real **distinto** del de hoy, a propósito — consulta las reglas de ese notebook).

## Tarea / Ideas de práctica

1. Cambia el `area` de la Parte 4 a una región real distinta (p. ej., el Mediterráneo, o aguas cercanas a tu propio país) y vuelve a ejecutar el notebook — ¿se parece la relación viento-oleaje de la Parte 7?
2. Implementa la partición espacial más estricta sugerida en la Parte 9: divide por un umbral de longitud (p. ej., entrena con todo lo que esté al oeste de -5°, prueba con todo lo que esté al este) en vez de una partición aleatoria — ¿cuánto cambia el R² reportado?
3. Añade un pequeño MLP (al estilo de `NB11`) a la comparación de la Parte 9 — ¿supera aquí al Random Forest, y coincide eso con la expectativa general de `NB17` o la contradice?
4. En la Parte 11, prueba un año reservado distinto como objetivo de predicción genuino (p. ej., `2018-01-15`, añadido o intercambiado con el conjunto de entrenamiento) — ¿se mantiene consistente la precisión de la predicción a través de distintos años reservados?
5. Amplía la solicitud de la Parte 4 a una pequeña ventana de días alrededor del 15 de enero (p. ej., 10–20 de enero) para cada año de entrenamiento, en vez de un único día — ¿cambia notablemente el dato adicional por año los resultados de la Parte 10 o de la Parte 11?
6. Usando el mapa de residuos de la Parte 10 y los mapas de predicho frente a real de la Parte 11, identifica la región donde más lucha la *predicción genuina* (Parte 11), y propón (en una celda markdown, sin necesidad de código) una característica que podría explicar ese error.

> ***Como siempre: un caso de estudio nuevo solo está "resuelto" en cuanto puedes explicar tanto lo que el modelo acertó como dónde tuvo dificultades — y, según la Parte 11, solo en cuanto lo has probado sobre algo que nunca vio venir, de una forma que sea realmente una prueba justa.***